In [1]:
from pathlib import Path
import sys
import json
from datetime import datetime
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
SELECTED_MODEL_PATH = (
    PROJECT_ROOT
    / "reports"
    / "selected_model.json"
)

THRESHOLD_PATH = (
    PROJECT_ROOT
    / "reports"
    / "decision_threshold.json"
)


with open(
    SELECTED_MODEL_PATH,
    "r"
) as file:
    selected_info = json.load(file)


with open(
    THRESHOLD_PATH,
    "r"
) as file:
    threshold_info = json.load(file)


selected_family = (
    selected_info["model_family"]
)

selected_params = (
    selected_info["best_parameters"]
)

decision_threshold = (
    threshold_info["selected_threshold"]
)


print(
    "Model:",
    selected_family
)

print(
    "Threshold:",
    decision_threshold
)

Model: XGBoost
Threshold: 0.4825977087020874


In [4]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "telco_churn_clean.csv"
)


df = pd.read_csv(DATA_PATH)

print(
    df.shape
)

(7043, 21)


In [5]:
X_full = df.drop(
    columns=[
        "customerID",
        "Churn",
    ]
)


y_full = df["Churn"].map({
    "No":0,
    "Yes":1
})


print(
    X_full.shape
)

print(
    y_full.shape
)

(7043, 19)
(7043,)


In [6]:
from src.features.preprocessing import build_preprocessor


def build_final_pipeline(
    family,
    parameters
):

    if family == "Logistic Regression":

        model = LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            random_state=42
        )


    elif family == "Random Forest":

        model = RandomForestClassifier(
            random_state=42,
            n_jobs=1
        )


    elif family == "XGBoost":

        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=1
        )


    else:
        raise ValueError(
            "Unknown model"
        )


    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                build_preprocessor()
            ),

            (
                "classifier",
                model
            )
        ]
    )


    pipeline.set_params(
        **parameters
    )


    return pipeline

In [7]:
# Fit final production pipeline
production_pipeline = (
    build_final_pipeline(
        selected_family,
        selected_params
    )
)


production_pipeline.fit(
    X_full,
    y_full
)


print(
    "Production pipeline trained"
)

Production pipeline trained


In [8]:
#create model folder
MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)

MODEL_DIR.mkdir(
    exist_ok=True
)

In [9]:
#save complete pipeline
PIPELINE_PATH = (
    MODEL_DIR
    / "churn_prediction_pipeline.joblib"
)


joblib.dump(
    production_pipeline,
    PIPELINE_PATH
)


print(
    "Saved:",
    PIPELINE_PATH
)

Saved: c:\AI_Projects\customer-churn-ml-system\models\churn_prediction_pipeline.joblib


In [10]:
#save metadata
metadata = {

    "project":
        "Customer Churn Prediction",

    "model_family":
        selected_family,

    "selection_metric":
        "Average Precision",

    "decision_threshold":
        float(
            decision_threshold
        ),

    "training_rows":
        int(
            len(X_full)
        ),

    "features":
        list(
            X_full.columns
        ),

    "created_at":
        str(
            datetime.now()
        )
}


METADATA_PATH = (
    MODEL_DIR
    / "model_metadata.json"
)


with open(
    METADATA_PATH,
    "w"
) as file:

    json.dump(
        metadata,
        file,
        indent=4
    )


print(
    "Metadata saved"
)

Metadata saved


In [11]:
#Save feature schema
schema = {

    "required_features":
        list(
            X_full.columns
        ),

    "feature_count":
        len(
            X_full.columns
        )
}


SCHEMA_PATH = (
    MODEL_DIR
    / "feature_schema.json"
)


with open(
    SCHEMA_PATH,
    "w"
) as file:

    json.dump(
        schema,
        file,
        indent=4
    )


print(
    "Schema saved"
)

Schema saved


In [12]:
#reload dataset
loaded_pipeline = joblib.load(
    PIPELINE_PATH
)


print(
    type(loaded_pipeline)
)

<class 'sklearn.pipeline.Pipeline'>


In [13]:
#real customer prediction test
sample_customer = X_full.iloc[
    [0]
]


sample_probability = (
    loaded_pipeline
    .predict_proba(
        sample_customer
    )[:,1][0]
)


sample_prediction = int(
    sample_probability
    >= decision_threshold
)


print(
    "Probability:",
    round(
        sample_probability,
        4
    )
)


print(
    "Prediction:",
    sample_prediction
)

Probability: 0.7994
Prediction: 1


In [14]:
#batch prediction test
sample_batch = X_full.head(10)


probabilities = (
    loaded_pipeline
    .predict_proba(
        sample_batch
    )[:,1]
)


predictions = (
    probabilities
    >= decision_threshold
).astype(int)


result = pd.DataFrame({

    "Probability":
        probabilities,

    "Prediction":
        predictions
})


result

,Probability,Prediction
0,0.799408,1
1,0.087683,0
2,0.493788,1
3,0.088781,0
4,0.807871,1
5,0.874692,1
6,0.612019,1
7,0.344758,0
8,0.722393,1
9,0.062492,0


In [16]:
#Validate artifact
assert PIPELINE_PATH.exists()

assert METADATA_PATH.exists()

assert SCHEMA_PATH.exists()

assert len(
    metadata["features"]
) == 19


assert (
    0
    <
    decision_threshold
    <
    1
)


print(
    "Production artifact validation passed"
)

Production artifact validation passed
